In [ ]:
# package import
import chipwhisperer as cw
from cw_plugins.scopes import *
from cw_plugins.targets import *
import gui_helper as gui

## Scope Setup
### Connect to the ChipWhisperer Scope
This notebook uses a ChipWhisperer **Lite** or **Husky** capture board. Run the next cell to connect with `cw.scope()`. For an external VISA oscilloscope, use [acquire_traces_with_visa_scope.ipynb](acquire_traces_with_visa_scope.ipynb).


In [ ]:
scope = cw.scope()

### Configure Sampling and Trigger Settings
Run the next cell to display the settings panel. **Detected board** shows the capture board identified by `cw.scope()`; the target FPGA board is selected separately below.

- **Sampling Rate (MSa/s)**: desired sampling rate. Husky uses an ADC multiplier of 1 internally; no multiplier input is required.
- **Trace Length**: total waveform duration, including pretrigger samples (default: **10 us**).
- **Pretrigger**: duration recorded before the trigger; it must be smaller than Trace Length.
- **Delay**: nonnegative capture offset after the trigger.
- **Gain (dB)**: analog input gain.
- **Trigger Pins**: trigger input, fr example `tio4`. Selected TIO pins are configured as high-impedance inputs.
- **Trigger Mode**: `rising_edge` by default; `falling_edge`, `high`, and `low` are also available. Husky uses the `basic` trigger module.
- **Timeout (s)**: maximum wait for a capture trigger.

Trace Length, Pretrigger, and Delay each accept `s`, `ms`, `us`, or `ns`.

Click **apply** to configure the scope.

In [ ]:
gui.showCWScopeConfig(scope)

## Target Hardware Setup
Run the next cell to display the target settings panel. Select `SAKURA-X` or `CW305`, then choose the implementation that matches the FPGA design:

- `AES RTL`: AES-128 RTL core; select `aist`, `google`, or `rsm` under **Select RTL Core**.
- `AES HLS`: AES-128 implementation generated by high-level synthesis.
- `AES VexRiscV`: AES-128 software running on VexRiscV; **Enable Masking** selects the masked version.

### SAKURA-X
Program the board with the matching shell and target design before connecting. Select both **Data Port** (FT2232H Channel A) and **Reset Port** (Channel B). These must be different ports belonging to the same board.

Alternatively, enable **Auto detect** on Linux with the repository's udev rules installed and one SAKURA-X board connected. On other platforms, select both ports explicitly. The GUI does not program the SAKURA-X FPGA.

### CW305
If you do not want to use the preinstalled bitstream, use **Bitfile** to select a `.bit` file and **HWH file** to select the corresponding `.hwh` memory-map file. For a custom design, provide matching files and select the matching implementation. If files are omitted, the target driver uses its default file selection; the bundled examples include prebuilt bitstreams and HWH files.

See the [hardware documentation](../docs/hardware.md) for board preparation and target-specific details.


In [ ]:
gui.showBoardSettingsPanel()

### Connect to the Target
After completing the settings above, run the next cell to connect the selected target and assign it to `target`. Finish scope configuration before starting capture.

For SAKURA-X, `gui.connectBoard(scope)` does not perform a hardware reset. If the target is stuck, call `target.hard_reset()` after connecting and before capture to reset the shell controller and target hardware.


In [ ]:
target = gui.connectBoard(scope)

## Trace Capture and Save
Run the next cell to display the capture panel.

- **Number of Traces**: number of successful traces to collect (default: 100).
- **Draw Interval**: redraw the waveform after this many successful traces (default: 5).
- **Change Key**: generate a new key before capture. The displayed key is fixed throughout a capture; plaintexts are generated for each trace.

Click **Start Capture** to acquire traces. Failed captures are retried until the requested number of successful traces is reached. Each plotted waveform is the most recently collected trace.

When capture finishes, select a save directory, enter **Project Name**, and click **Save Project**. Enable **Overwrite if exists** to replace an existing project. Saving creates a ChipWhisperer `.cwp` project and its associated data directory. Save the current capture before starting another capture, which creates a new in-memory project.


In [ ]:
gui.showCapturePanel(scope, target)